# Oil Price Model Exploration

Interactive analysis of the oil-futures forecasting pipeline: forward curves, the crude→pump lag model, backtest performance, and war-cost scenarios.

**Data sources:** `data/oil-futures.json`, `data/lag-model.json`, `data/oil-backtest.json`

In [ ]:
import json, math
from pathlib import Path
from datetime import datetime, timezone
import numpy as np

DATA = Path("..") / "data"

with open(DATA / "oil-futures.json") as f:
    oil = json.load(f)
with open(DATA / "lag-model.json") as f:
    lag = json.load(f)
with open(DATA / "oil-backtest.json") as f:
    bt = json.load(f)

print(f"Futures updated:  {oil['last_updated']}")
print(f"Lag model fitted: {lag['generated']}")
print(f"Backtest run:     {bt['generated']}")
print(f"Brent contracts:  {len(oil['brent'])}  |  WTI contracts: {len(oil['wti'])}")
print(f"Historical Brent: {len(oil['historical_brent'])} months")
print(f"Historical pump:  {len(oil['historical_pump'])} months")
print(f"Snapshots:        {len(oil['snapshots'])}")

## 1. Forward Curves: Brent & WTI

Current futures strip — what the market expects crude prices to do over the next year.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

fig, ax = plt.subplots(figsize=(12, 5))

for series, color, label in [('brent', '#2563eb', 'Brent'), ('wti', '#dc2626', 'WTI')]:
    contracts = oil[series]
    dates = [datetime.strptime(c['expiry'], '%Y-%m') for c in contracts]
    prices = [c['price'] for c in contracts]
    ax.plot(dates, prices, 'o-', color=color, label=f'{label} ({len(contracts)} contracts)', markersize=4)

ax.set_title('Futures Forward Curves', fontsize=14, fontweight='bold')
ax.set_ylabel('Price ($/bbl)')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
fig.autofmt_xdate()
ax.legend()
ax.grid(True, alpha=0.3)

# Annotate front-month spread
brent_front = oil['brent'][0]['price']
wti_front = oil['wti'][0]['price']
ax.annotate(f'Brent-WTI spread: ${brent_front - wti_front:.2f}',
            xy=(0.02, 0.95), xycoords='axes fraction', fontsize=10,
            bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow', alpha=0.8))

plt.tight_layout()
plt.show()

print(f"Front-month Brent: ${brent_front:.2f}  |  WTI: ${wti_front:.2f}")
print(f"Back-month Brent:  ${oil['brent'][-1]['price']:.2f}  |  WTI: ${oil['wti'][-1]['price']:.2f}")
print(f"Backwardation (Brent): ${brent_front - oil['brent'][-1]['price']:.2f}")
print(f"Backwardation (WTI):   ${wti_front - oil['wti'][-1]['price']:.2f}")

### Historical Snapshots

How the forward curve has shifted over recent weeks.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

cmap = plt.cm.viridis
snapshots = oil.get('snapshots', [])
n_snap = len(snapshots)

for idx, snap in enumerate(snapshots):
    color = cmap(idx / max(n_snap - 1, 1))
    for ax, key in [(ax1, 'brent'), (ax2, 'wti')]:
        contracts = snap[key]
        dates = [datetime.strptime(c['expiry'], '%Y-%m') for c in contracts]
        prices = [c['price'] for c in contracts]
        ax.plot(dates, prices, '-', color=color, alpha=0.7, label=snap['date'])

for ax, title in [(ax1, 'Brent Snapshots'), (ax2, 'WTI Snapshots')]:
    # Add current curve
    series_key = 'brent' if 'Brent' in title else 'wti'
    dates = [datetime.strptime(c['expiry'], '%Y-%m') for c in oil[series_key]]
    prices = [c['price'] for c in oil[series_key]]
    ax.plot(dates, prices, 'k-', linewidth=2, label='Current')
    ax.set_title(title, fontweight='bold')
    ax.set_ylabel('$/bbl')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %y'))
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

fig.autofmt_xdate()
plt.tight_layout()
plt.show()

## 2. Lag Model: Crude → Pump Pass-Through

The PDL(degree=2) model with 8 weekly lags maps Brent crude ($/gal) to retail gasoline price.

$$\text{pump}_t = \alpha + \sum_{k=0}^{8} \beta_k \cdot \text{crude}_{t-k} + \varepsilon_t$$

In [ ]:
betas = lag['betas']
cum = lag['cumulative_passthrough']
n_lags = len(betas)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Beta coefficients
ax = axes[0]
colors = ['#2563eb' if b >= 0 else '#dc2626' for b in betas]
ax.bar(range(n_lags), betas, color=colors, edgecolor='white', linewidth=0.5)
ax.axhline(0, color='black', linewidth=0.5)
ax.set_xlabel('Lag (weeks)')
ax.set_ylabel('Beta coefficient')
ax.set_title('PDL Beta Weights', fontweight='bold')
ax.set_xticks(range(n_lags))
for i, b in enumerate(betas):
    ax.text(i, b + 0.01 * (1 if b >= 0 else -1), f'{b:.3f}', ha='center', va='bottom' if b >= 0 else 'top', fontsize=8)

# Cumulative pass-through
ax = axes[1]
ax.plot(range(n_lags), cum, 'o-', color='#2563eb', markersize=6)
ax.axhline(1.0, color='gray', linestyle='--', alpha=0.5, label='Full pass-through')
ax.fill_between(range(n_lags), cum, alpha=0.15, color='#2563eb')
ax.set_xlabel('Lag (weeks)')
ax.set_ylabel('Cumulative pass-through')
ax.set_title('Cumulative Pass-Through', fontweight='bold')
ax.set_xticks(range(n_lags))
ax.legend()

# PDL polynomial shape
ax = axes[2]
x = np.linspace(0, n_lags - 1, 100)
# Fit degree-2 polynomial to the betas
poly_coeffs = np.polyfit(range(n_lags), betas, 2)
poly_vals = np.polyval(poly_coeffs, x)
ax.plot(x, poly_vals, '-', color='#7c3aed', linewidth=2, label=f'Polynomial fit')
ax.plot(range(n_lags), betas, 'ko', markersize=6, label='Actual betas')
ax.axhline(0, color='black', linewidth=0.5)
ax.set_xlabel('Lag (weeks)')
ax.set_ylabel('Beta')
ax.set_title('PDL Polynomial Shape (degree=2)', fontweight='bold')
ax.set_xticks(range(n_lags))
ax.legend()

plt.tight_layout()
plt.show()

print(f"Model: {lag['method']}, n_obs={lag['n_obs']}, R²={lag['r2']:.4f}")
print(f"RMSE: ${lag['rmse']:.4f}/gal  |  MAE: ${lag['mae']:.4f}/gal")
print(f"Intercept (alpha): ${lag['alpha']:.4f}")
print(f"Total pass-through: {lag['total_passthrough']:.4f}")
print(f"Seasonal premium (Apr-Sep): ${lag['seasonal_premium']:.4f}/gal")
print(f"Polynomial coefficients: {[f'{c:.6f}' for c in poly_coeffs]}")

### NARDL Asymmetric Pass-Through

The NARDL model splits crude price changes into positive and negative components to capture the "rockets and feathers" effect — gas stations raise prices faster than they lower them.

In [ ]:
betas_pos = lag['betas_pos']
betas_neg = lag['betas_neg']

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Side-by-side beta comparison
ax = axes[0]
x = np.arange(len(betas_pos))
w = 0.35
ax.bar(x - w/2, betas_pos, w, color='#dc2626', alpha=0.8, label=f'Positive (up) — total: {lag["total_passthrough_pos"]:.4f}')
ax.bar(x + w/2, betas_neg, w, color='#2563eb', alpha=0.8, label=f'Negative (down) — total: {lag["total_passthrough_neg"]:.4f}')
ax.axhline(0, color='black', linewidth=0.5)
ax.set_xlabel('Lag (weeks)')
ax.set_ylabel('Beta')
ax.set_title('NARDL: Asymmetric Betas', fontweight='bold')
ax.set_xticks(x)
ax.legend(fontsize=9)

# Cumulative comparison
ax = axes[1]
cum_pos = np.cumsum(betas_pos)
cum_neg = np.cumsum(betas_neg)
cum_sym = lag['cumulative_passthrough']
ax.plot(range(len(cum_pos)), cum_pos, 'o-', color='#dc2626', label='NARDL positive (price rises)')
ax.plot(range(len(cum_neg)), cum_neg, 's-', color='#2563eb', label='NARDL negative (price falls)')
ax.plot(range(len(cum_sym)), cum_sym, '^--', color='gray', alpha=0.6, label='Symmetric PDL')
ax.axhline(1.0, color='black', linestyle=':', alpha=0.3)
ax.set_xlabel('Lag (weeks)')
ax.set_ylabel('Cumulative pass-through')
ax.set_title('Cumulative Pass-Through: Asymmetric vs Symmetric', fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Asymmetry test
asym = lag['asymmetry_test']
wald = lag['wald_test']
print(f"Asymmetry test: coeff={asym['coeff']:.4f}, t={asym['t_stat']:.3f}, p={asym['p_approx']:.4f} → {'Significant' if asym['significant'] else 'Not significant'}")
print(f"Wald test (sum_pos ≠ sum_neg): W={wald['W_stat']:.3f}, p={wald['p_approx']:.4f} → {'Significant' if wald['significant'] else 'Not significant'}")
print(f"\nNARDL R²: {lag['nardl_r2']:.4f} vs PDL R²: {lag['r2']:.4f}  (improvement: {lag['nardl_r2'] - lag['r2']:.4f})")
print(f"NARDL RMSE: ${lag['nardl_rmse']:.4f} vs PDL RMSE: ${lag['rmse']:.4f}")

## 3. Backtest Performance

20 years of out-of-sample forecasting accuracy — WTI crude price predictions vs realized spot.

In [ ]:
# Summary table for all horizons
print("=" * 75)
print(f"{'Horizon':<12} {'N':>6} {'Bias':>8} {'MAE':>8} {'RMSE':>8} {'Naive':>8} {'R²':>6} {'±10%':>6} {'±20%':>6}")
print("-" * 75)
for h in ['1m', '2m', '3m', '4m']:
    r = bt['results'][h]
    print(f"{r['label']:<12} {r['n']:>6} {r['bias']:>8.2f} {r['mae']:>8.2f} {r['rmse']:>8.2f} {r['naive_rmse']:>8.2f} {r['r2']:>6.2f} {r['within_10pct']:>5.0%} {r['within_20pct']:>5.0%}")
print("=" * 75)

print("\nBand Calibration:")
print(f"{'Horizon':<12} {'68% cov':>10} {'95% cov':>10} {'Above 1σ':>10} {'Below 1σ':>10}")
print("-" * 55)
for h in ['1m', '2m', '3m', '4m']:
    r = bt['results'][h]
    print(f"{r['label']:<12} {r['coverage_68']:>9.0%} {r['coverage_95']:>9.0%} {r['pct_above_upper1']:>9.0%} {r['pct_below_lower1']:>9.0%}")

### Forecast Error Over Time (1-month & 4-month)

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

for ax, horizon, color in [(axes[0], '1m', '#2563eb'), (axes[1], '4m', '#7c3aed')]:
    monthly = bt['results'][horizon]['monthly']
    months = [datetime.strptime(m['month'], '%Y-%m') for m in monthly]
    errors = [m['error'] for m in monthly]
    pct_errors = [m['pct_error'] for m in monthly]
    
    ax.bar(months, pct_errors, width=25, color=color, alpha=0.6)
    ax.axhline(0, color='black', linewidth=0.5)
    ax.set_ylabel('% Error')
    ax.set_title(f'{bt["results"][horizon]["label"]} Horizon — Monthly % Error', fontweight='bold')
    ax.grid(True, alpha=0.2)
    
    # Shade regime periods
    regimes = bt['results']['regimes']
    crisis_regimes = [r for r in regimes if 'crisis' in r['name'].lower() or 'crash' in r['name'].lower()]
    for r in crisis_regimes:
        start = datetime.strptime(r['start'], '%Y-%m')
        end = datetime.strptime(r['end'], '%Y-%m')
        ax.axvspan(start, end, alpha=0.1, color='red', label=r['name'])

axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
axes[1].xaxis.set_major_locator(mdates.YearLocator(2))
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

### Performance by Market Regime

In [ ]:
regimes = bt['results']['regimes']

# 1-month regime table (from pre-computed data)
print("1-MONTH HORIZON — Performance by Market Regime")
print("=" * 85)
print(f"{'Regime':<25} {'N':>6} {'Bias':>8} {'MAE':>8} {'RMSE':>8} {'Avg |%err|':>12}")
print("-" * 85)
for r in regimes:
    print(f"{r['name']:<25} {r['n']:>6} {r['bias']:>8.2f} {r['mae']:>8.2f} {r['rmse']:>8.2f} {r['pct_error']:>11.2f}%")

# 4-month regime stats (computed from monthly data)
monthly_4m = bt['results']['4m']['monthly']
print(f"\n4-MONTH HORIZON — Performance by Market Regime")
print("=" * 85)
print(f"{'Regime':<25} {'N':>6} {'Bias':>8} {'MAE':>8} {'RMSE':>8} {'Avg |%err|':>12}")
print("-" * 85)
for r in regimes:
    entries = [m for m in monthly_4m if m['month'] >= r['start'] and m['month'] <= r['end']]
    if not entries:
        continue
    n = len(entries)
    bias = sum(e['error'] for e in entries) / n
    mae = sum(abs(e['error']) for e in entries) / n
    rmse = math.sqrt(sum(e['error'] ** 2 for e in entries) / n)
    pct = sum(abs(e['pct_error']) for e in entries) / n
    print(f"{r['name']:<25} {n:>6} {bias:>8.2f} {mae:>8.2f} {rmse:>8.2f} {pct:>11.2f}%")

In [ ]:
# Regime performance heatmap
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

regime_names = [r['name'] for r in regimes]

for ax, horizon, title in [(axes[0], '1m', '1-Month'), (axes[1], '4m', '4-Month')]:
    if horizon == '1m':
        maes = [r['mae'] for r in regimes]
        pcts = [r['pct_error'] for r in regimes]
    else:
        maes, pcts = [], []
        for r in regimes:
            entries = [m for m in monthly_4m if m['month'] >= r['start'] and m['month'] <= r['end']]
            if entries:
                maes.append(sum(abs(e['error']) for e in entries) / len(entries))
                pcts.append(sum(abs(e['pct_error']) for e in entries) / len(entries))
            else:
                maes.append(0)
                pcts.append(0)
    
    y = np.arange(len(regime_names))
    bars = ax.barh(y, pcts, color=plt.cm.RdYlGn_r(np.array(pcts) / max(pcts)), edgecolor='white')
    ax.set_yticks(y)
    ax.set_yticklabels(regime_names, fontsize=9)
    ax.set_xlabel('Average |% Error|')
    ax.set_title(f'{title} Horizon', fontweight='bold')
    ax.invert_yaxis()
    
    for i, (p, m) in enumerate(zip(pcts, maes)):
        ax.text(p + 0.3, i, f'{p:.1f}% (MAE ${m:.1f})', va='center', fontsize=8)

plt.tight_layout()
plt.show()

### Annual MAE Trend

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5))

colors_h = {'1m': '#2563eb', '2m': '#059669', '3m': '#d97706', '4m': '#7c3aed'}
for h in ['1m', '2m', '3m', '4m']:
    by_year = bt['results'][h]['by_year']
    years = [y['year'] for y in by_year]
    maes = [y['mae'] for y in by_year]
    ax.plot(years, maes, 'o-', color=colors_h[h], label=bt['results'][h]['label'], markersize=4)

ax.set_xlabel('Year')
ax.set_ylabel('MAE ($/bbl)')
ax.set_title('Annual Mean Absolute Error by Horizon', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

# Mark crisis years
for year, label in [(2008, 'GFC'), (2020, 'COVID')]:
    ax.axvline(year, color='red', linestyle=':', alpha=0.4)
    ax.text(year + 0.1, ax.get_ylim()[1] * 0.9, label, fontsize=8, color='red')

plt.tight_layout()
plt.show()

### Pump Price Backtest (Lag Model)

In [ ]:
print("PUMP PRICE BACKTEST (crude → retail gasoline via lag model)")
print("=" * 80)
print(f"{'Horizon':<12} {'N':>6} {'Bias':>8} {'MAE':>8} {'RMSE':>8} {'R²':>6} {'68% cov':>8} {'95% cov':>8}")
print("-" * 80)
for h in ['1m', '2m', '3m', '4m']:
    r = bt['pump_results'][h]
    print(f"{r['label']:<12} {r['n']:>6} {r['bias']:>7.2f}¢ {r['mae']:>7.2f}¢ {r['rmse']:>7.2f}¢ {r['r2']:>6.2f} {r['coverage_68']:>7.0%} {r['coverage_95']:>7.0%}")

# Pump predicted vs realized scatter
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, h, title in [(axes[0], '1m', '1-Month'), (axes[1], '4m', '4-Month')]:
    monthly = bt['pump_results'][h]['monthly']
    pred = [m['pred'] for m in monthly]
    real = [m['realized'] for m in monthly]
    
    ax.scatter(pred, real, s=15, alpha=0.5, color='#2563eb')
    mn, mx = min(min(pred), min(real)), max(max(pred), max(real))
    ax.plot([mn, mx], [mn, mx], 'k--', alpha=0.4, label='Perfect forecast')
    ax.set_xlabel('Predicted pump ($/gal)')
    ax.set_ylabel('Realized pump ($/gal)')
    ax.set_title(f'{title} — Pump Forecast vs Realized', fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.set_aspect('equal')

plt.tight_layout()
plt.show()

## 4. Futures → Pump Price Conversion

Using the lag model to convert the current Brent forward curve into predicted retail gasoline prices.

In [ ]:
def interpolate_crude(contracts, weeks_out):
    """Interpolate Brent price at a given weeks-out point on the forward curve."""
    if weeks_out <= 0:
        return contracts[0]['price']
    for i in range(len(contracts) - 1):
        expiry_i = datetime.strptime(contracts[i]['expiry'] + '-15', '%Y-%m-%d')
        expiry_j = datetime.strptime(contracts[i+1]['expiry'] + '-15', '%Y-%m-%d')
        weeks_i = (expiry_i - datetime.now()).days / 7
        weeks_j = (expiry_j - datetime.now()).days / 7
        if weeks_i <= weeks_out <= weeks_j:
            frac = (weeks_out - weeks_i) / (weeks_j - weeks_i) if weeks_j != weeks_i else 0
            return contracts[i]['price'] + frac * (contracts[i+1]['price'] - contracts[i]['price'])
    return contracts[-1]['price']

def predict_pump(contracts, expiry, alpha, betas, seasonal_premium):
    """Convert a forward curve into a pump price prediction for a given delivery month."""
    delivery_date = datetime.strptime(expiry + '-15', '%Y-%m-%d')
    delivery_weeks = (delivery_date - datetime.now()).days / 7
    
    weighted_crude_gal = 0
    for k, beta in enumerate(betas):
        crude_bbl = interpolate_crude(contracts, delivery_weeks - k)
        weighted_crude_gal += beta * (crude_bbl / 42)  # bbl → gal
    
    delivery_month = int(expiry[5:7])
    is_summer = 4 <= delivery_month <= 9
    
    return alpha + weighted_crude_gal + (seasonal_premium if is_summer else 0)

# Compute pump forecast for each Brent contract
alpha = lag['alpha']
betas_pdl = lag['betas']
sp = lag['seasonal_premium']

print(f"{'Month':<12} {'Brent':>8} {'Crude/gal':>10} {'Pump':>8} {'Season':>8}")
print("-" * 52)
for c in oil['brent']:
    pump = predict_pump(oil['brent'], c['expiry'], alpha, betas_pdl, sp)
    crude_gal = c['price'] / 42
    month_num = int(c['expiry'][5:7])
    season = "Summer" if 4 <= month_num <= 9 else "Winter"
    print(f"{c['label']:<12} ${c['price']:>7.2f} ${crude_gal:>8.3f} ${pump:>7.3f}  {season}")

## 5. What-If Scenarios

Sensitivity analysis: what happens to pump prices if crude moves $10, $20, $30/bbl from the current front-month?

In [ ]:
baseline_brent = oil['brent'][0]['price']
baseline_expiry = oil['brent'][0]['expiry']
baseline_pump = predict_pump(oil['brent'], baseline_expiry, alpha, betas_pdl, sp)

shocks = [-30, -20, -10, 0, 10, 20, 30]
pump_prices = []

for shock in shocks:
    # Shift entire curve by the shock amount
    shifted = [{'expiry': c['expiry'], 'price': c['price'] + shock, 'label': c['label']} for c in oil['brent']]
    pump = predict_pump(shifted, baseline_expiry, alpha, betas_pdl, sp)
    pump_prices.append(pump)

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#dc2626' if s > 0 else '#2563eb' if s < 0 else '#374151' for s in shocks]
bars = ax.bar([f'{"+" if s > 0 else ""}{s}' for s in shocks], pump_prices, color=colors, alpha=0.8)
ax.axhline(baseline_pump, color='gray', linestyle='--', alpha=0.5, label=f'Current: ${baseline_pump:.2f}/gal')
ax.set_xlabel('Crude Price Shock ($/bbl)')
ax.set_ylabel('Predicted Pump Price ($/gal)')
ax.set_title('Pump Price Sensitivity to Crude Oil Shocks', fontweight='bold')
ax.legend()

for bar, price in zip(bars, pump_prices):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.03,
            f'${price:.2f}', ha='center', fontsize=9)

plt.tight_layout()
plt.show()

print(f"\nBaseline: Brent ${baseline_brent:.2f}/bbl → Pump ${baseline_pump:.2f}/gal")
print(f"Pass-through rate: ~${lag['total_passthrough']/42:.3f}/gal per $1/bbl crude change")
print(f"  = ~{lag['total_passthrough']/42 * 100:.1f}¢/gal per $1/bbl")

## 6. War-Cost Scenario Calculator

Estimate the additional cost of gasoline per household given a crude oil price premium from geopolitical conflict. The EIA estimates average US household drives ~12,000 miles/year at ~25 MPG = 480 gal/year.

In [ ]:
GAL_PER_YEAR = 480  # ~12,000 mi / 25 MPG
PASSTHROUGH_PER_GAL = lag['total_passthrough'] / 42  # $/gal per $1/bbl

# Pre-war baseline: assume Brent was ~$75/bbl (typical 2024 level)
PRE_WAR_BRENT = 75.0

# Current and scenario Brent levels
scenarios = {
    'Pre-conflict baseline':  PRE_WAR_BRENT,
    'Current futures':        baseline_brent,
    'Moderate escalation':    baseline_brent + 15,
    'Major supply disruption': baseline_brent + 40,
    'Iran strait closure':    baseline_brent + 60,
}

print(f"Crude → pump pass-through: {PASSTHROUGH_PER_GAL:.4f} $/gal per $/bbl")
print(f"Household consumption: {GAL_PER_YEAR} gal/year")
print()
print(f"{'Scenario':<28} {'Brent':>8} {'Pump Δ':>10} {'Annual Cost':>14}")
print("=" * 65)

baseline_pump_ref = alpha + PASSTHROUGH_PER_GAL * PRE_WAR_BRENT * 42 / 42 + sp  # simplified steady-state
for name, brent in scenarios.items():
    pump_delta = (brent - PRE_WAR_BRENT) * PASSTHROUGH_PER_GAL
    annual_cost = pump_delta * GAL_PER_YEAR
    sign = '+' if pump_delta >= 0 else ''
    print(f"{name:<28} ${brent:>7.2f} {sign}${pump_delta:>7.2f}/gal  {sign}${annual_cost:>8.0f}/yr")

# Visualization
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# Left: pump price delta per gallon
names = list(scenarios.keys())
brents = list(scenarios.values())
deltas = [(b - PRE_WAR_BRENT) * PASSTHROUGH_PER_GAL for b in brents]
annual = [d * GAL_PER_YEAR for d in deltas]

colors = ['#059669' if d <= 0 else '#d97706' if d < 0.5 else '#dc2626' for d in deltas]
ax1.barh(names, deltas, color=colors)
ax1.set_xlabel('Pump Price Change ($/gal)')
ax1.set_title('Impact on Gas Price per Gallon', fontweight='bold')
ax1.axvline(0, color='black', linewidth=0.5)
for i, d in enumerate(deltas):
    ax1.text(d + 0.02, i, f'{"+" if d >= 0 else ""}{d:.2f}', va='center', fontsize=9)

# Right: annual household cost
ax2.barh(names, annual, color=colors)
ax2.set_xlabel('Additional Annual Cost ($)')
ax2.set_title('Annual Household Gas Cost Impact', fontweight='bold')
ax2.axvline(0, color='black', linewidth=0.5)
for i, a in enumerate(annual):
    ax2.text(a + 10, i, f'{"+" if a >= 0 else ""}${a:.0f}', va='center', fontsize=9)

plt.tight_layout()
plt.show()

## 7. Historical Context: Crude vs Pump

3-year history of Brent crude and national average pump prices, showing the lagged relationship the model captures.

In [ ]:
hist_brent = oil['historical_brent']
hist_pump = oil['historical_pump']

fig, ax1 = plt.subplots(figsize=(13, 5))

# Brent on left axis
dates_b = [datetime.strptime(h['month'], '%Y-%m') for h in hist_brent]
prices_b = [h['price'] for h in hist_brent]
ax1.plot(dates_b, prices_b, 'o-', color='#2563eb', markersize=4, label='Brent crude ($/bbl)')
ax1.set_ylabel('Brent Crude ($/bbl)', color='#2563eb')
ax1.tick_params(axis='y', labelcolor='#2563eb')

# Pump on right axis
ax2 = ax1.twinx()
dates_p = [datetime.strptime(h['month'], '%Y-%m') for h in hist_pump]
prices_p = [h['price'] for h in hist_pump]
ax2.plot(dates_p, prices_p, 's-', color='#dc2626', markersize=4, label='Pump price ($/gal)')
ax2.set_ylabel('Pump Price ($/gal)', color='#dc2626')
ax2.tick_params(axis='y', labelcolor='#dc2626')

ax1.set_title('Historical Brent Crude vs US Average Pump Price (3 years)', fontweight='bold')
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax1.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
fig.autofmt_xdate()

# Combined legend
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')

ax1.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()

# Correlation
if len(hist_brent) == len(hist_pump):
    corr = np.corrcoef(prices_b, prices_p)[0, 1]
    print(f"Brent/pump correlation (monthly): {corr:.3f}")
print(f"Latest Brent: ${prices_b[-1]:.2f}/bbl  |  Latest pump: ${prices_p[-1]:.2f}/gal")

## 8. Crack Spreads

The refining margin (crack spread) is the difference between refined product prices and crude input. Seasonal patterns affect summer driving season vs winter.

In [ ]:
crack = oil.get('crack_spreads', {})
by_month = crack.get('by_month', {})
avg = crack.get('average', 0)

if by_month:
    months_sorted = sorted(by_month.keys())
    month_labels = [datetime.strptime(f'2024-{m}', '%Y-%m').strftime('%b') for m in months_sorted]
    spreads = [by_month[m] for m in months_sorted]
    
    fig, ax = plt.subplots(figsize=(10, 5))
    colors = ['#dc2626' if 4 <= int(m) <= 9 else '#2563eb' for m in months_sorted]
    ax.bar(month_labels, spreads, color=colors, alpha=0.8)
    ax.axhline(avg, color='gray', linestyle='--', label=f'Average: ${avg:.2f}/bbl')
    ax.set_ylabel('Crack Spread ($/bbl)')
    ax.set_title('Monthly Crack Spreads (Red = Summer, Blue = Winter)', fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.2)
    
    for i, (s, m) in enumerate(zip(spreads, months_sorted)):
        ax.text(i, s + 0.3, f'${s:.1f}', ha='center', fontsize=8)
    
    plt.tight_layout()
    plt.show()
    
    summer = [by_month[m] for m in months_sorted if 4 <= int(m) <= 9]
    winter = [by_month[m] for m in months_sorted if not (4 <= int(m) <= 9)]
    print(f"Average crack spread: ${avg:.2f}/bbl")
    print(f"Summer avg (Apr-Sep): ${np.mean(summer):.2f}/bbl")
    print(f"Winter avg (Oct-Mar): ${np.mean(winter):.2f}/bbl")
    print(f"Seasonal spread: ${np.mean(summer) - np.mean(winter):.2f}/bbl")
else:
    print(f"Crack spread data: average = ${avg:.2f}/bbl (monthly breakdown not available)")